In [0]:
df = spark.table("workspace.default.lc_accepted_raw")

# Distribución de la variable loan_status (como un PROC FREQ en SAS)
display(df.groupBy("loan_status").count().orderBy("count", ascending=False))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Iniciar sesión y crear datos de prueba (Simulando SAS)
spark = SparkSession.builder.appName("ProcFreq_SAS").getOrCreate()

# 2. Paso 1: Agrupar por la variable y contar frecuencias básicas
df_freq = df.groupBy("loan_status").agg(F.count("*").alias("Frequency"))

# 3. Paso 2: Definir ventanas para calcular totales y acumulados
# Ventana global (sin partición ni orden) para el total general de filas
ventana_total = Window.partitionBy()

# Ventana ordenada para calcular las sumas acumuladas (Equivalente al orden de renglones en SAS)
# Usamos .rowsBetween() para garantizar que sume fila por fila de forma acumulativa
ventana_acumulada = Window.orderBy(F.col("Frequency").desc())\
                          .rowsBetween(Window.unboundedPreceding, Window.currentRow)

# 4. Construir la tabla final con porcentajes y acumulados
df_proc_freq = df_freq \
    .withColumn("Percent", (F.col("Frequency") / F.sum("Frequency").over(ventana_total)) * 100) \
    .withColumn("Cumulative_Frequency", F.sum("Frequency").over(ventana_acumulada)) \
    .withColumn("Cumulative_Percent", F.sum("Percent").over(ventana_acumulada)) \
    .select(
        "loan_status",
        F.round("Frequency", 2).alias("Frequency"),
        F.round("Percent", 2).alias("Percent"),
        F.round("Cumulative_Frequency", 2).alias("Cumulative Frequency"),
        F.round("Cumulative_Percent", 2).alias("Cumulative Percent")
    )

# Mostrar el resultado final ordenado por mayor frecuencia
df_proc_freq.show()

# Estructura estilo PROC TABULATE
df_tabulate = df.groupBy("loan_status") \
    .pivot("grade") \
    .agg(
        F.count("loan_status").alias("Total_Creditos"),
        F.round(F.mean("loan_amnt"), 2).alias("Monto_Promedio")
    )

# En Databricks, display() te ordenará las columnas perfectamente
display(df_tabulate)

# Esto te funcionará exactamente como en Pandas
df.limit(5).toPandas().head()


In [0]:
from pyspark.sql import functions as F

dfp = (df.filter(F.col("loan_status").isNotNull())                  # quita las 33 filas basura
         .withColumn("issue_date", F.to_date("issue_d", "MMM-yyyy"))  # "Dec-2015" -> fecha
         .withColumn("issue_year", F.year("issue_date")))

# Estado del crédito por año de originación y plazo (como un PROC TABULATE)
display(dfp.groupBy("issue_year", "term")
           .pivot("loan_status")
           .count()
           .orderBy("issue_year", "term"))


In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.default.lc_accepted_raw")

MALO  = ["Charged Off", "Default", "Late (31-120 days)"]
BUENO = ["Fully Paid"]
INDET = ["In Grace Period", "Late (16-30 days)"]

# Variables auxiliares: fecha de originación, año, plazo y clase B/G/I
base = (df.withColumn("issue_date", F.to_date("issue_d", "MMM-yyyy"))
          .withColumn("issue_year", F.year("issue_date"))
          .withColumn("term_months", F.regexp_extract("term", r"(\d+)", 1).cast("int"))  # " 36 months" -> 36
          .withColumn("gbi", F.when(F.col("loan_status").isin(MALO), "B")
                              .when(F.col("loan_status").isin(BUENO), "G")
                              .when(F.col("loan_status").isin(INDET), "I")
                              .when(F.col("loan_status") == "Current", "C")
                              .otherwise("X")))   # NULL y "Does not meet the credit policy"

# Cascada de exclusiones: cada paso filtra sobre el anterior
pasos = [
    ("0. Base total",                          F.lit(True)),
    ("1. Excluye filas basura (status NULL)",  F.col("loan_status").isNotNull()),
    ("2. Excluye política de crédito anterior", F.col("gbi") != "X"),
    ("3. Solo plazo 36 meses",                 F.col("term_months") == 36),
    ("4. Solo cosechas 2011-2015",             F.col("issue_year").between(2011, 2015)),
    ("5. Excluye Current (sin desempeño)",     F.col("gbi") != "C"),
    ("6. Excluye indeterminados",              F.col("gbi") != "I"),
]

cascada, p = [], base
for nombre, condicion in pasos:
    p = p.filter(condicion)
    cascada.append((nombre, p.count()))

display(spark.createDataFrame(cascada, ["paso", "casos"]))

# Población final: objetivo binario y marca de muestra
pob = (p.withColumn("target", F.when(F.col("gbi") == "B", 1).otherwise(0))
        .withColumn("sample", F.when(F.col("issue_year") <= 2014, "DEV").otherwise("OOT")))

pob.write.mode("overwrite").saveAsTable("workspace.default.lc_population")

# Bad rate por muestra y cosecha
display(spark.table("workspace.default.lc_population")
        .groupBy("sample", "issue_year")
        .agg(F.count("*").alias("casos"),
             F.sum("target").alias("malos"),
             F.round(F.avg("target") * 100, 2).alias("bad_rate_pct"))
        .orderBy("issue_year"))
